<a href="https://colab.research.google.com/github/dennisddschulz/cas-artificial-intelligence/blob/main/08_drl_einstieg/01_DRL_Einstieg_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DRL Intro: MDP, Return, V/Q/Advantage – Monte-Carlo Evaluation (FrozenLake)

Ziel:
1) Episode sammeln (Rollout)
2) Returns G_t berechnen
3) Monte-Carlo Schätzung von V(s) und Q(s,a)
4) Advantage A(s,a) berechnen
5) Aus Q eine ε-greedy Policy ableiten
6) Zeigen, dass sich V(start) verbessert (Policy Improvement)


In [79]:
!pip -q install gymnasium

import gymnasium as gym
import numpy as np
from collections import defaultdict

SEED = 43
rng = np.random.default_rng(SEED)


In [80]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

s0, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n

print("nS:", nS, "nA:", nA, "start:", s0)

nS: 16 nA: 4 start: 0


In [81]:
traj = rollout_episode(env, pi_eps, max_steps=200, seed=SEED + 100)
print("New episode length with improved policy:", len(traj))
print("First steps of new trajectory:", traj[:8])

New episode length with improved policy: 2
First steps of new trajectory: [(0, 1, 0), (4, 2, 0)]


Now that the `traj` variable has been updated with a trajectory from the improved policy, please re-run the animation cell (`VQzcNJKLAw7I`) to visualize it. You should now see a longer and potentially more successful path!

## Policy und Rollout

- Policy: Funktion, die aus Zustand s eine Aktion a wählt.
- Rollout: wir lassen Agent+Env laufen und speichern (s, a, r) pro Schritt.


In [82]:
def random_policy(s, nA):
    return int(rng.integers(nA))

def rollout_episode(env, policy_fn, max_steps=200, seed=0):
    traj = []  # list of (s, a, r)
    s, _ = env.reset(seed=seed)
    for _ in range(max_steps):
        a = policy_fn(s, env.action_space.n)
        s2, r, terminated, truncated, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if terminated or truncated:
            break
    return traj

traj = rollout_episode(env, random_policy, seed=SEED)
print("episode length:", len(traj))
print("first steps:", traj[:8])


episode length: 5
first steps: [(0, 0, 0), (0, 1, 0), (4, 3, 0), (0, 1, 0), (4, 2, 0)]


## Returns berechnen

Return G_t ist die discounted Summe der zukünftigen Rewards ab Schritt t.
Wir berechnen das rückwärts:
G = 0
G <- r + gamma * G


In [83]:
def compute_returns(traj, gamma=0.99):
    G = 0.0
    returns = []
    # HA - Warum reversed?
    for (s, a, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

gamma = 0.99
Gs = compute_returns(traj, gamma=gamma)
list(zip(traj[:8], Gs[:8]))


[((0, 0, 0), 0.0),
 ((0, 1, 0), 0.0),
 ((4, 3, 0), 0.0),
 ((0, 1, 0), 0.0),
 ((4, 2, 0), 0.0)]

## MC Evaluation von V(s)

First-Visit MC:
- pro Episode zählt nur das erste Auftreten eines Zustands s
- V(s) = Durchschnitt der beobachteten Returns in s

```
seen = set()
for t, (s, a, r) in enumerate(traj):
    if s in seen:
        continue
    seen.add(s)
    V[s] += G[t]
```

Every-Visit MC:
- pro Episode zählt jedes Auftreten eines Zustands s
- V(s) ist der Durchschnitt der beobachteten Return über alle Besuche von s in allen Episoden.
```
for t, (s, a, r) in enumerate(traj):
    V[s] += G[t]
```


In [100]:
# This function is being removed as its functionality is merged into mc_evaluate_V with an every_visit flag.

In [101]:
def mc_evaluate_V(env, policy_fn, episodes=3000, gamma=0.99, seed=0, first_visit=True):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        if first_visit:
            seen = set()

        for t, (s, a, r) in enumerate(traj):
            if first_visit:
                if s in seen:
                    continue
                seen.add(s)
            returns_sum[s] += Gs[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V

V_rand = mc_evaluate_V(env, random_policy, episodes=4000, gamma=gamma, seed=SEED, first_visit=True)

s0, _ = env.reset(seed=SEED)
print("V_random(start) (First-Visit):", round(V_rand.get(s0, 0.0), 4))

V_random(start) (First-Visit): 0.0147


In [94]:
V_rand

{0: 0.010768418164540708,
 1: 0.009312191390857618,
 2: 0.015104381830296016,
 4: 0.014874605800623525,
 8: 0.031040732114055836,
 9: 0.07810101200935614,
 10: 0.1193015199071322,
 6: 0.02575523714317353,
 3: 0.007602648402526592,
 13: 0.16717000156632425,
 14: 0.4091957385106071}

## MC Evaluation von Q(s,a)

Analog:
- wir mitteln Returns pro (s,a)
- daraus können wir greedy / ε-greedy Policies bauen


In [95]:
def mc_evaluate_Q(env, policy_fn, episodes=6000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen_sa = set()
        for t, (s, a, r) in enumerate(traj):
            key = (s, a)
            if key in seen_sa:
                continue
            seen_sa.add(key)
            returns_sum[key] += Gs[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_count}
    return Q

Q_rand = mc_evaluate_Q(env, random_policy, episodes=8000, gamma=gamma, seed=SEED)
print("Q entries:", list(Q_rand.items())[:5])


Q entries: [((0, 2), 0.010359885047156129), ((1, 0), 0.012591689694625194), ((0, 3), 0.010998139312150374), ((0, 0), 0.00953038171569275), ((0, 1), 0.014252346994266692)]


## Advantage A(s,a)

A(s,a) = Q(s,a) - V(s)
Interpretation: wie viel besser/schlechter ist Aktion a gegenüber dem "Durchschnitt" in s.


In [96]:
def advantage(V, Q, s, a):
    return Q.get((s, a), 0.0) - V.get(s, 0.0)

# Advantage im Startzustand für alle Aktionen
adv_start = [(a, advantage(V_rand, Q_rand, s0, a)) for a in range(nA)]
adv_start


[(0, -0.0012380364488479567),
 (1, 0.003483928829725985),
 (2, -0.0004085331173845786),
 (3, 0.00022972114760966625)]

## Policy Improvement: ε-greedy aus Q

- greedy: a = argmax_a Q(s,a)
- ε-greedy: mit Wahrscheinlichkeit ε zufällig, sonst greedy

Dann evaluieren wir die neue Policy wieder mit MC und vergleichen V(start).


In [102]:
def epsilon_greedy_policy_from_Q(Q, nA, eps=0.1):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        qs = [Q.get((s, a), 0.0) for a in range(nA)]
        return int(np.argmax(qs))
    return policy

pi_eps = epsilon_greedy_policy_from_Q(Q_rand, nA, eps=0.1)

V_eps = mc_evaluate_V(env, pi_eps, episodes=4000, gamma=gamma, seed=SEED, first_visit=True) # Defaulting to First-Visit
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))
print("V_eps(start):   ", round(V_eps.get(s0, 0.0), 4))

V_random(start): 0.0147
V_eps(start):    0.8432


## Mini Loop: wiederholte Verbesserung (Iteration)

Wir wiederholen:
1) Q unter aktueller Policy schätzen
2) neue ε-greedy Policy bauen
3) V(start) loggen

Achtung: Das ist noch nicht "Policy Iteration" im strengen Sinn,
aber zeigt sehr gut die Grundidee: Bessere Wertschätzungen (V/Q) → bessere Entscheidungsgrundlage → verbesserte Policy


In [103]:
def policy_improvement_loop(env, init_policy, iters=5, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=0.99, seed=0, first_visit_V=True):
    policy = init_policy
    history = []

    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)
        policy = epsilon_greedy_policy_from_Q(Q, env.action_space.n, eps=eps)
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k, first_visit=first_visit_V)
        history.append((k, V.get(s0, 0.0)))

    return history

hist = policy_improvement_loop(env, random_policy, iters=6, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=gamma, seed=SEED, first_visit_V=True) # Defaulting to First-Visit for V evaluation
hist

[(0, 0.8522580719939679),
 (1, 0.8347298709905082),
 (2, 0.8522276277514498),
 (3, 0.30104908062558294),
 (4, 0.8464489432166935),
 (5, 0.19473130161119348)]

In [99]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# traj = list of (state, action, reward)
# env = your Gymnasium environment

frames = []

# Reset environment to start
obs, _ = env.reset()

# Collect frames by stepping through traj
for s, a, r in traj:
    obs, reward, terminated, truncated, _ = env.step(a)
    img = env.render()  # get RGB array
    frames.append(img)
    if terminated or truncated:
        break

# Create figure for animation
fig, ax = plt.subplots()
ax.axis('off')

# Check if frames is not empty before trying to display the first frame
if frames:
    im = ax.imshow(frames[0])
else:
    print("No frames collected for animation.")
    # Close the figure if no frames to avoid empty plot display
    plt.close(fig)
    # Return a dummy HTML or message if no animation can be made
    ani = None # Ensure ani is defined even if no frames

def update(frame):
    im.set_data(frame)
    return [im]

if frames: # Only create animation if frames exist
    ani = animation.FuncAnimation(fig, update, frames=frames, interval=500, blit=True)
    plt.close(fig) # Close the figure to prevent duplicate static plot
    display(HTML(ani.to_jshtml()))
else:
    print("Animation could not be created as no frames were collected.")

## Was haben wir heute gelernt?

- Reward vs Return: Return ist das Ziel, nicht der einzelne Reward.
- V(s) und Q(s,a) sind Erwartungswerte von Returns.
- Monte-Carlo schätzt diese Werte aus Episoden (ohne Modell von P).
- Advantage erklärt "wie gut ist diese Aktion relativ zum Durchschnitt in s".
- Aus Q kann man eine bessere Policy ableiten (ε-greedy).
